In [4]:
# Cell 1 — Read nested JSON and explode
from pyspark.sql import functions as F
from pyspark.sql import types as T

raw = spark.read \
    .option("multiline", "true") \
    .json("abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/raw/products/products_catalog.json")

# Products are inside a "data" array — explode it into rows
products_df = raw.select(F.explode(F.col("data")).alias("p")).select("p.*")

print("Products:", products_df.count())
products_df.printSchema()


StatementMeta(, 6185270e-e161-4174-80d8-165298736777, 6, Finished, Available, Finished, False)

Products: 200
root
 |-- base_price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- selling_price: double (nullable = true)
 |-- stock_quantity: long (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- weight_kg: double (nullable = true)



In [6]:
# Cell 2 — Clean and type
cleaned = products_df \
    .withColumn("base_price",    F.col("base_price").cast(T.DecimalType(12,2))) \
    .withColumn("selling_price", F.col("selling_price").cast(T.DecimalType(12,2))) \
    .withColumn("created_at",    F.to_timestamp(F.col("created_at"))) \
    .withColumn(
        # Flag where selling > base (data error)
        "price_anomaly",
        F.col("selling_price") > F.col("base_price")
    ) \
    .withColumn("silver_created_at", F.current_timestamp())

cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_products")

print("Done:", spark.sql("SELECT COUNT(*) FROM silver_products").collect()[0][0])

StatementMeta(, 6185270e-e161-4174-80d8-165298736777, 8, Finished, Available, Finished, False)

Done: 200
